# NLP Project, part 3: a search engine for the German corpus
**Group 2 (German)** · spaCy `de_core_news_md` · corpus from part 1, entity index from part 2
**Members:** Blevis Allushi · Darsi Ramaj · Geri Belba

**Application area: information retrieval.** The corpus (part 1) and the named-entity index (part 2) become a search engine: a user types a German query and gets the texts that answer it, ranked, with the matching words marked. German makes this harder than English in four ways, and each preprocessing step below answers one of them:

| German problem | Example | Our step |
|---|---|---|
| Inflection | *Flüsse* in the query, *Fluss* / *Flusses* in the text | spaCy lemmas |
| Compounds | *Wörterbuch* is inside *Rechtschreibwörterbuch* | compound splitter |
| Umlauts, ß, case | *muenchen*, *fussball* typed without umlauts | folding + query normalisation |
| Name variants | *Goethe*, *Johann Wolfgang von Goethe*, *Goethes* | entity linking (part 2) |

All code is in `src/sprache/search/` (analysis, index, evaluation) and `src/sprache/web/` (web app). This notebook calls it and discusses the results.

**Run:** Kernel → *Restart Kernel and Run All Cells* (about 1 minute), or `python -m sprache notebook`. Live demo: `python -m sprache serve`.

| Section | Content | Owner |
|---|---|---|
| 0-1 | Setup, the application | Blevis |
| 2 | Preprocessing for search | Blevis |
| 3-4 | Index and queries | Blevis |
| 5 | Evaluation and error analysis | Blevis |
| 6-7 | Demo, discussion | Blevis |

## 0 · Setup

In [1]:
import sys, time
from pathlib import Path
try:
    import sprache
except ImportError:
    sys.path.insert(0, str(Path('../src').resolve()))
    import sprache

import pandas as pd
from IPython.display import Markdown, display

from sprache import analysis, config, corpus
from sprache.ner.kb import fold
from sprache.search import evaluate as ev
from sprache.search.engine import SearchEngine, parse_corpus
from sprache.search.text import ANALYZERS, tokens

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 200)
nlp = analysis.load_model()
print('spaCy   :', __import__('spacy').__version__, '·', f"{nlp.meta['lang']}_{nlp.meta['name']} {nlp.meta['version']}")
print('sprache :', sprache.__version__)

spaCy   : 3.8.18 · de_core_news_md 3.8.0
sprache : 3.0.0


## 1 · The application: what the engine has to do

Input: a German query of a few words, typed the way people type (lower case, no umlauts, typos).
Output: the most relevant of the 100 texts, ranked, each with a snippet and its named entities.

Design decisions:
* **Ranking: BM25** over an inverted index, the standard bag-of-words ranking function. It rewards query words that are frequent in the text and rare in the corpus, and normalises for text length. Article titles count three times.
* **The same analysis on both sides.** Documents and queries go through the same `Analyzer`, so they meet in the same term space. Each step can be switched off, which is how section 5 measures it.
* **Entities as search terms.** Every linked entity mention from part 2 adds a term like `@Q64` (Berlin) to its text. A query that names an entity in any form reaches all texts that mention it.
* **Cleaned corpus.** We index `corpus_clean/`: the IPA transcriptions and dangling labels removed in part 1 would otherwise become index terms.

In [2]:
t = time.time()
docs = parse_corpus(nlp)                       # spaCy on all texts (cached in models/ after the first run)
engine = SearchEngine(nlp, docs)               # full system
print(f'{len(docs)} texts indexed in {time.time() - t:.1f} s')

100 texts indexed in 0.9 s


## 2 · Preprocessing for search

### 2.1 One sentence through every step
Each column adds one step to the column before it. Terms in brackets are compound parts (half weight); `·` means the token produces no term.

In [3]:
sent = 'Der Duden, das bekannteste Rechtschreibwörterbuch Deutschlands, erschien in größeren Städten zur Energiewende.'
toks = tokens(nlp(sent))
table = []
for i, t in enumerate(toks):
    row = {'token': t.text}
    for a in ANALYZERS:
        terms = [term if w == 1 else f'({term})'
                 for j, term, w in a.terms(toks, engine.splitter if a.compounds else None, engine.names) if j == i]
        row[a.name] = ' '.join(terms) or '·'
    table.append(row)
pd.DataFrame(table).set_index('token')

,surface forms,+ stop words removed,+ lemmas,+ umlaut/case folding,+ compound splitting
token,,,,,
Der,der,·,·,·,·
Duden,duden,duden,duden,duden,duden
",",·,·,·,·,·
das,das,·,·,·,·
bekannteste,bekannteste,bekannteste,bekannt,bekannt,bekannt
Rechtschreibwörterbuch,rechtschreibwörterbuch,rechtschreibwörterbuch,rechtschreibwörterbuch,rechtschreibwoerterbuch,rechtschreibwoerterbuch (woerterbuch)
Deutschlands,deutschlands,deutschlands,deutschland,deutschland,deutschland
",",·,·,·,·,·
erschien,erschien,erschien,erscheinen,erscheinen,erscheinen


* **Stop words** (*Der*, *das*, *in*, *zur*) and punctuation disappear. They would match almost every text.
* **Lemmas** turn *Städten* into *Stadt*, *erschien* into *erscheinen* and *größeren* into *groß*, so an inflected form in a query finds every form in the text.
* **Folding** turns *groß* into *gross*, so a query typed without umlauts or ß finds it. It is the same key the knowledge base uses in part 2 (`ner.kb.fold`).
* **Compound splitting** adds the parts in brackets with half weight: *Rechtschreibwörterbuch* also counts as *wörterbuch*, *Energiewende* as *energie* and *wende*. The compound itself stays a term, so an exact query still ranks highest.
* **Names keep their form.** spaCy's lemmatizer damages names it mistakes for plural nouns (it turns *Beethoven* into *Beethove*), so a word the knowledge base knows as a name keeps its form and only loses a genitive *-s* (*Deutschlands* → *deutschland*).
* The lemmatizer is not perfect on common nouns either: in some contexts it leaves plurals such as *Flüsse* or *Weihnachtsmärkte* unchanged. Because queries and texts go through the same lemmatizer, many such errors cancel out; the back-off in section 4 catches some of the rest.

### 2.2 The compound splitter
German writes compounds as one word, and spaCy keeps them as one token with the whole word as lemma (part 1, section 6). Our splitter is **corpus-driven**: its vocabulary is every word form and lemma in the corpus. A word is split into the fewest known parts, allowing the linking elements (*Fugenelemente*) *-s-, -es-, -n-, -en-, -e-, -er-*. Ties go to the longest last part, because the head carries the meaning: a *Weihnachtsmarkt* is a *Markt*. Parts are mapped back to lemmas (*weihnacht* → *weihnachten*).

In [4]:
words = ['Weihnachtsmarkt', 'Energiewende', 'Bundesliga', 'Landeshauptstadt', 'Klimawandel', 'Kernenergie',
         'Stadtverwaltung', 'Rechtschreibwörterbuch', 'Heidelberg', 'Bodensee', 'Oktoberfest']
pd.DataFrame([{'word': w, 'split': ' + '.join(engine.splitter.split(fold(w))) or '-',
               'index terms added': ', '.join(engine.splitter.parts(fold(w))) or '-'} for w in words])

,word,split,index terms added
0,Weihnachtsmarkt,weihnacht + markt,"weihnachten, markt"
1,Energiewende,energie + wende,"energie, wende"
2,Bundesliga,bundes + liga,"bund, liga"
3,Landeshauptstadt,landes + hauptstadt,"land, hauptstadt"
4,Klimawandel,klima + wandel,"klima, wandel"
5,Kernenergie,kern + energie,"kern, energie"
6,Stadtverwaltung,stadt + verwaltung,"stadt, verwaltung"
7,Rechtschreibwörterbuch,-,woerterbuch
8,Heidelberg,-,-
9,Bodensee,-,-


* *Rechtschreibwörterbuch* cannot be split completely (*rechtschreib* is not a corpus word), so a **back-off** keeps the known head *wörterbuch*.
* *Heidelberg* is not split: *heidel* is no word. Names (`PROPN`) are never split in the index anyway, which protects place names whose parts happen to be words.
* The splitter is only as good as its vocabulary: a 21,000-word corpus does not contain every possible part.

In [5]:
n_tok = sum(1 for d in docs.values() for t in d.toks if t.pos in ('NOUN', 'ADJ'))
n_split = sum(1 for d in docs.values() for t in d.toks if t.pos in ('NOUN', 'ADJ') and engine.splitter.split(fold(t.lemma)))
print(f'{n_split} of {n_tok} noun/adjective tokens ({n_split / n_tok:.0%}) are split into known parts')

934 of 7600 noun/adjective tokens (12%) are split into known parts


## 3 · The index

In [6]:
idx = engine.index
word_terms = [t for t in idx.postings if not t.startswith('@')]
entity_terms = [t for t in idx.postings if t.startswith('@')]
print(f'{len(word_terms):,} word terms, {len(entity_terms):,} entity terms (Wikidata items)')
print(f'average text length: {idx.avg_len:.0f} weighted terms')
df = pd.Series({t: len(p) for t, p in idx.postings.items()}, name='texts')
print('\nterms in the most texts (lowest idf):')
print(df.sort_values(ascending=False).head(8).to_string())

5,625 word terms, 591 entity terms (Wikidata items)
average text length: 146 weighted terms

terms in the most texts (lowest idf):
deutsch        56
@Q183          47
deutschland    47
reich          44
land           37
gelten         31
zahl           31
welt           30


After stop-word removal the most widespread terms are content words typical of encyclopedic text. BM25's idf gives them a small weight automatically. The entity term for Germany (`@Q183`) is among them: it appears in most texts.

## 4 · Queries

### 4.1 How a query is read
Four steps run before the ranking. The web app shows the same information next to every result list.
1. **Normalisation** (part 2, `ner.noise.Normaliser`): truecasing, umlaut restoration and typo repair from the corpus lexicon.
2. **spaCy**: lemma and part of speech.
3. **Same analyzer** as the documents, with a back-off chain for words the index does not know: lemma → word form (spaCy lemmatises a lone query word without context) → compound parts.
4. **Entity linking** with the knowledge base: names become Wikidata ids.

In [7]:
for q in ['muenchen bier', 'Beethofen', 'Goethes Werke', 'Habsburgerreich']:
    qa = engine.analyse_query(q)
    print(f'{q!r:20} -> {qa.normalised!r:24} corrections={qa.corrections}')
    print(f'{"":24}terms={list(qa.terms)}  entities={[(e["text"], e["qid"], e["label"]) for e in qa.entities]}\n')

'muenchen bier'      -> 'München Bier'           corrections=[('muenchen', 'München'), ('bier', 'Bier')]
                        terms=['muenchen', 'bier', '@Q1726']  entities=[('München', 'Q1726', 'München')]

'Beethofen'          -> 'Beethoven'              corrections=[('Beethofen', 'Beethoven')]
                        terms=['beethoven']  entities=[]

'Goethes Werke'      -> 'Goethe Werke'           corrections=[('Goethes', 'Goethe')]
                        terms=['goethe', 'werk', '@Q5879']  entities=[('Goethe', 'Q5879', 'Johann Wolfgang von Goethe')]

'Habsburgerreich'    -> 'Habsburgerreich'        corrections=[]
                        terms=['habsburgerreich', '@Q153136']  entities=[('Habsburgerreich', 'Q153136', 'Habsburgermonarchie')]



* *muenchen bier*: the lexicon restores *München* and the capital of *Bier*; *München* is linked to its Wikidata item.
* *Beethofen*: not in the lexicon, but exactly one lexicon entry is one edit away.
* *Goethes Werke*: the genitive is removed from the name, and the name is linked to the person.
* *Habsburgerreich* does not occur in the corpus as a word, but the knowledge base knows it as a name of the Habsburg monarchy, so entity linking finds the texts that mention the monarchy in any form.

### 4.2 Results

In [8]:
def show(q, k=5):
    qa, results = engine.search(q, k)
    display(Markdown(f'**{q}**' + (f' → *{qa.normalised}*' if qa.corrections else '')))
    rows = []
    for r in results:
        snip = ''.join(f'[{s["text"]}]' if s['hit'] else s['text'] for s in r['snippet'])
        rows.append({'title': r['title'], 'score': r['score'], 'snippet ([..] = match)': snip[:160]})
    display(pd.DataFrame(rows))

show('Flüsse in Deutschland')
show('Wörterbuch', 3)

**Flüsse in Deutschland**

,title,score,snippet ([..] = match)
0,Rhein,5.88,… Ab dem [Zusammenfluss] der beiden Hauptquelläste Vorderrhein und Hinterrhein wird der [Fluss] meist in f...
1,Mosel,5.85,Die Mosel (lateinisch Mosella) ist mit 544 km Fließstrecke der zweitlängste [Nebenfluss] des Rheins nach d...
2,Donau,5.48,"… Er durchfließt bzw. berührt dabei zehn Länder ([Deutschland], Österreich, Slowakei, Ungarn, Kroatien, Se..."
3,Elbe,4.91,… Gemessen an der Größe ihres Einzugsgebietes von 148.300 km² liegt die Elbe für Mitteleuropa gemäß Liste ...
4,Hamburg,4.87,… Typisch für das Stadtbild sind weiterhin viele Klinkerfassaden sowie die Wassernähe mit zahlreichen [Flü...


**Wörterbuch**

,title,score,snippet ([..] = match)
0,Duden,5.77,Der Duden ist ein [Rechtschreibwörterbuch] der deutschen Sprache. …


*Flüsse in Deutschland* returns the river articles first. They match through the lemma *Fluss* (the word form *Flüsse* is rare in them) and through the entity *Deutschland*. *Wörterbuch* finds the Duden through the compound *Rechtschreibwörterbuch*.

## 5 · Evaluation

### 5.1 Test queries
We wrote 30 queries and judged, for each one, which texts are relevant by reading them (`data/search/queries.tsv`): grade 2 = the text is about the query, grade 1 = it answers part of it. The queries are grouped by the German problem they test.

In [9]:
queries = ev.load_queries()
pd.DataFrame([{'id': q.id, 'query': q.text, 'kind': q.kind,
               'relevant': ', '.join(f'{docs[f].title} ({g})' for f, g in q.relevant.items())} for q in queries])

,id,query,kind,relevant
0,q01,Flüsse in Deutschland,inflection,"Rhein (2), Donau (2), Elbe (2), Mosel (2)"
1,q02,Komponisten,inflection,"Ludwig van Beethoven (2), Johann Sebastian Bach (2), Wolfgang Amadeus Mozart (2)"
2,q03,erneuerbare Energien,inflection,Energiewende (2)
3,q04,Philosophen,inflection,"Immanuel Kant (2), Karl Marx (2)"
4,q05,deutsche Schriftsteller,inflection,"Johann Wolfgang von Goethe (2), Franz Kafka (2), Thomas Mann (2)"
5,q06,Inseln in der Ostsee,inflection,"Rügen (2), Ostsee (2)"
6,q07,Fall der Mauer 1989,plain,"Berliner Mauer (2), Deutsche Wiedervereinigung (2), Berlin (1)"
7,q08,Bundeskanzler,plain,"Angela Merkel (2), Konrad Adenauer (2), Otto von Bismarck (1)"
8,q09,Gebirge,compound,"Harz (Mittelgebirge) (2), Alpen (2), Thüringer Wald (2), Zugspitze (1)"
9,q10,Mundart,plain,"Dialekt (2), Deutsche Sprache (1)"


### 5.2 Ablation: what each step adds
Each system adds one step to the one before. Metrics, averaged over the queries:
* **P@5**: share of the top 5 that is relevant (low by design: most queries have only 1-3 relevant texts);
* **R@10**: share of the relevant texts in the top 10;
* **MRR**: 1 / rank of the first relevant text;
* **nDCG@10**: graded, rewards relevant texts at the top; 1.0 is a perfect ranking.

In [10]:
systems = ev.systems(nlp, docs, engine.kb, engine.normaliser)
overall, by_kind = ev.ablation(nlp, queries, systems)
overall

,P@5,R@10,MRR,nDCG@10
system,,,,
surface forms,0.280,0.591,0.744,0.626
+ stop words removed,0.280,0.591,0.742,0.625
+ lemmas,0.327,0.673,0.800,0.720
+ umlaut/case folding,0.387,0.812,0.933,0.859
+ compound splitting,0.420,0.877,0.933,0.892
+ entity linking (Tutorial 2),0.413,0.877,0.933,0.897
+ query normalisation (full system),0.420,0.910,0.967,0.931


In [11]:
by_kind

,compound,entity,inflection,plain,spelling
system,,,,,
surface forms,0.795,0.909,0.652,0.745,0.000
+ stop words removed,0.795,0.909,0.652,0.743,0.000
+ lemmas,0.845,0.909,0.907,0.828,0.000
+ umlaut/case folding,0.845,0.909,0.966,0.828,0.750
+ compound splitting,0.945,0.960,0.966,0.828,0.797
+ entity linking (Tutorial 2),0.944,0.960,0.966,0.847,0.797
+ query normalisation (full system),0.944,0.960,0.966,0.847,0.997


Reading the tables:
* **Stop-word removal** changes almost nothing. BM25's idf already gives *der*, *und* or *in* a weight close to zero, so removing them mainly makes the index smaller.
* **Lemmas** give the largest gain on the *inflection* queries (*Flüsse*, *Philosophen*, *Komponisten*).
* **Folding** is the largest single step overall. It makes the umlaut-less queries (*zuerich*, *muenchen*, *fussball*) work at all, and it also merges case variants.
* **Compound splitting** lifts the *compound* queries (*Wörterbuch*, *Weihnachten*, *Ostsee* in *Ostseeküste*).
* **Entity linking** adds little on these queries, because most names are also found as words. Its value shows in the result view (entity chips) and in name variants.
* **Query normalisation** fixes the last *spelling* query: the typo *Beethofen*.

### 5.3 Per query, and the errors

In [12]:
full = ev.per_query(engine, queries)
base = ev.per_query(systems[0][1], queries)
full.insert(4, 'baseline nDCG@10', base['nDCG@10'])
full.round(2)

,id,query,kind,P@5,baseline nDCG@10,R@10,MRR,nDCG@10,top 3
0,q01,Flüsse in Deutschland,inflection,0.8,0.39,1.00,1.0,1.00,"Rhein, Mosel, Donau"
1,q02,Komponisten,inflection,0.6,1.00,1.00,1.0,1.00,"Johann Sebastian Bach, Ludwig van Beethoven, Wolfgang Amadeus Mozart"
2,q03,erneuerbare Energien,inflection,0.2,1.00,1.00,1.0,1.00,"Energiewende, Photosynthese, Glühlampe"
3,q04,Philosophen,inflection,0.4,0.00,1.00,1.0,1.00,"Immanuel Kant, Karl Marx"
4,q05,deutsche Schriftsteller,inflection,0.4,0.77,0.67,1.0,0.77,"Thomas Mann, Franz Kafka, Deutsche Sprache"
5,q06,Inseln in der Ostsee,inflection,0.4,0.61,1.00,1.0,1.00,"Ostsee, Rügen, Nordsee"
6,q07,Fall der Mauer 1989,plain,0.6,0.95,1.00,1.0,0.95,"Berliner Mauer, Berlin, Deutsche Wiedervereinigung"
7,q08,Bundeskanzler,plain,0.6,1.00,1.00,1.0,1.00,"Konrad Adenauer, Angela Merkel, Otto von Bismarck"
8,q09,Gebirge,compound,0.6,0.91,1.00,1.0,0.99,"Thüringer Wald, Harz (Mittelgebirge), Alpen"
9,q10,Mundart,plain,0.4,0.83,1.00,1.0,1.00,"Dialekt, Deutsche Sprache"


In [13]:
full[full['nDCG@10'] < 0.9][['id', 'query', 'kind', 'nDCG@10', 'top 3']].round(2)

,id,query,kind,nDCG@10,top 3
4,q05,deutsche Schriftsteller,inflection,0.77,"Thomas Mann, Franz Kafka, Deutsche Sprache"
11,q12,Schweiz,plain,0.88,"Zürich, Bodensee, Kleingarten"
13,q14,Radioaktivität,plain,0.00,Karl Marx
14,q15,elektrischer Strom,plain,0.80,"Glühlampe, Elektrizität, Donau"
18,q19,Weihnachten,compound,0.83,Weihnachtsmarkt
29,q30,Habsburger,entity,0.88,"Dreißigjähriger Krieg, Wien"


Error analysis of the queries below 0.9:
* **Radioaktivität** (0.0): the texts say *radioaktiv* and *radioaktive Strahlung*. That is **derivation** (adjective → noun), not inflection: neither the lemmatizer nor the compound splitter connects the two, and the head back-off finds *Aktivität*, which is wrong. A stemmer or a derivation lexicon would fix this; we left it as a known limit.
* **deutsche Schriftsteller**: *Goethe* is called *Dichter* in his text, not *Schriftsteller*. A synonym list or word embeddings would be needed.
* **Weihnachten**: the Bach article mentions the *Weihnachtsoratorium*, but *Oratorium* is not a corpus word, so the compound is not split.
* **Habsburger**: *habsburgischen* (adjective) in the Beethoven text is not connected to the name: the same derivation problem.

**Caveat.** The query set is small and was written and judged by us, the people who built the system. The absolute numbers are therefore optimistic. The *differences* between the steps are the meaningful result, because every system was measured on the same queries.

## 6 · Demo and presentation

The web app runs locally with the standard library only (no extra packages):

```
python -m sprache serve          # opens http://127.0.0.1:8000
```

| View | What it shows |
|---|---|
| Present | The project as slides with live numbers (arrow keys, F for full screen) |
| Search | The engine: results with marked snippets and entity chips, how the query was read, a side-by-side comparison with the baseline |
| Pipeline | Any text through cleaning, spaCy, index terms and NER (BIO), including noisy input |
| Evaluation | The tables of section 5 |
| Corpus | All 100 texts with their entities, source and licence |

From the terminal: `python -m sprache search "Flüsse in Deutschland"` and `python -m sprache search-eval`.

## 7 · Discussion

* **German needs German preprocessing.** Lemmas, folding and compound splitting together give the largest part of the gain over plain word matching (section 5.2); stop-word removal alone adds nothing measurable.
* **The earlier parts pay off.** The cleaning of part 1 keeps IPA and template debris out of the index; the knowledge base and normaliser of part 2 give entity linking and spelling repair without any new data.
* **Limits.** 100 encyclopedic texts; queries judged by the developers; no derivation or synonyms (*Radioaktivität / radioaktiv*, *Dichter / Schriftsteller*); the entity index inherits the errors of the NER (about 0.8 F1 on names), e.g. single words like *Staaten* linked to the USA.
* **Next steps.** A larger corpus, a derivation-aware stemmer (e.g. Snowball) as a back-off, word embeddings for synonyms, and queries judged by people outside the team.

### Deliverables

| Requirement | Where |
|---|---|
| 1. Corpus collection | part 1 (`01_corpus.ipynb`), `corpus/`, `sources.csv` |
| 2. NLP preprocessing | part 1 sections 3-6 (cleaning, spaCy), part 2 (tokenisation, NER), section 2 here (lemmas, folding, compounds) |
| 3. Application area | this notebook: a search engine, evaluated in section 5 |
| 4. Weekly progress | git history of the repository |
| 5. Presentation | `python -m sprache serve` (Present view), this notebook, `docs/report.pdf` |